### ITEM PRICES 

In [1]:
import pandas as pd

In [17]:
df = pd.read_excel(r"C:\Users\nivet\Downloads\Purchase.xlsx")
df.head()

,Date,Particulars,Stock,Party GSTIN/UIN,Voucher Type,Invoice Type,Vch No.,Doc No.,Doc,Taxable,IGST,CGST,SGST,Invoice
0,2024-04-04,ABC Marble and Granites,Granites Sq Feet,33XXXXXXXXXXXXX,Inward Register,Item Invoice,ABC/012/2024-25,ABC/012/2024-25,2024-04-04,6000.0,NaN,540.00,540.00,7080.00
1,2024-04-06,Guru Granite,Granites Sq Feet,34XXXXXXXXXXXXX,Inward Register,Item Invoice,GURU/024/2024-25,GURU/024/2024-25,2024-04-06,33628.0,NaN,3026.52,3026.52,39681.04
2,2024-04-06,Guru Granite,Granites Sq Feet,34XXXXXXXXXXXXX,Inward Register,Item Invoice,GURU/018/2024-25,GURU/018/2024-25,2024-04-06,14402.0,NaN,1296.18,1296.18,16994.36
3,2024-04-06,G Marble,Granites Sq Feet,33XXXXXXXXXXXZK,Inward Register,Item Invoice,G/312/2024-25,G/312/2024-25,2024-04-06,3414.0,NaN,307.26,307.26,4028.52
4,2024-04-09,Guru Granite,Granites Sq Feet,34XXXXXXXXXXXXX,Inward Register,Item Invoice,GURU/812/2024-25,GURU/812/2024-25,2024-04-09,17332.0,NaN,1559.88,1559.88,20451.76


In [ ]:
import pandas as pd

def process_purchase_to_tally(input_file, output_file):
    df = pd.read_excel(input_file)

    # Fix the date column
    df['Date'] = pd.to_datetime(df['Date'].astype(str).str[:10])
    
    df = df.rename(columns={'Vch No.': 'voucher_no'})
    # Group by each voucher
    grouped = df.groupby('voucher_no')

    result = []

    for voucher_no, group in grouped:
        row = group.iloc[0]
        voucher_date = row['Date'].date()
        reference_no = row['voucher_no']
        ledger_name = row['Particulars']
        taxable = float(row['Taxable']) if pd.notna(row['Taxable']) else 0.0
        cgst = float(row['CGST']) if pd.notna(row['CGST']) else 0.0
        sgst = float(row['SGST']) if pd.notna(row['SGST']) else 0.0
        igst = float(row['IGST']) if pd.notna(row['IGST']) else 0.0
        total_amount = taxable + cgst + sgst + igst

        # Cr entry
        result.append({
            "Voucher Date": voucher_date,
            "Voucher Type Name": "Inward Register",
            "Reference No.": reference_no,
            "Ledger Name": ledger_name,
            "Ledger Amount": round(total_amount, 2),
            "Ledger Amount Dr/Cr": "Cr",
            "Change Mode": "Accounting Invoice"
        })

        # Dr entries
        result.append({
            "Voucher Date": "",
            "Voucher Type Name": "",
            "Reference No.": "",
            "Ledger Name": "PURCHASE",
            "Ledger Amount": round(taxable, 2),
            "Ledger Amount Dr/Cr": "Dr",
            "Change Mode": ""
        })

        result.append({
            "Voucher Date": "",
            "Voucher Type Name": "",
            "Reference No.": "",
            "Ledger Name": "INPUT CGST",
            "Ledger Amount": round(cgst, 2),
            "Ledger Amount Dr/Cr": "Dr",
            "Change Mode": ""
        })

        result.append({
            "Voucher Date": "",
            "Voucher Type Name": "",
            "Reference No.": "",
            "Ledger Name": "INPUT SGST",
            "Ledger Amount": round(sgst, 2),
            "Ledger Amount Dr/Cr": "Dr",
            "Change Mode": ""
        })

        result.append({
            "Voucher Date": "",
            "Voucher Type Name": "",
            "Reference No.": "",
            "Ledger Name": "INPUT IGST",
            "Ledger Amount": round(igst, 2),
            "Ledger Amount Dr/Cr": "Dr",
            "Change Mode": ""
        })

    # Convert to DataFrame and export
    result_df = pd.DataFrame(result)
    result_df.to_excel(output_file, index=False)


### Upload excel

In [14]:
# Example usage
input_csv = r"C:\Users\nivet\Downloads\GSTR-2A Reconciliation - Voucher Register.xlsx"
output_excel = r"C:\Users\nivet\Downloads\processed.xlsx"
process_purchase_to_tally(input_csv, output_excel)